# 01 : FHIR R4 → validation → SQLite

**Objectif.** Montrer le cœur du pipeline d’intégration sans dépendre de l’état changeant du serveur HAPI : extraction de ressources `Patient` depuis un `Bundle` représentatif, normalisation défensive, validation et persistance idempotente.

**Entrée :** un Bundle FHIR R4 fictif, sans donnée réelle.  
**Sortie :** une table SQLite temporaire contenant uniquement les patients valides.

Le module applicatif reste la source de vérité ; ce notebook l’importe et documente son comportement.


## 1. Configuration

On localise la racine du dépôt et on importe les fonctions existantes de parsing et de persistance. La base sera créée dans un dossier temporaire afin que l’exécution du notebook ne modifie aucun fichier métier.


In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.database import create_database, get_all_patients, save_patients
from src.parser import parse_patient, validate_patient

print(f"Racine du projet : {PROJECT_ROOT}")


Racine du projet : C:\Juliette Vanessa\Desktop\Stage\.audit-repos\healthcare-fhir-integration


> **Contrôle.** Les fonctions sont importées depuis `src/` : le notebook ne maintient pas une seconde implémentation du pipeline.


## 2. Bundle représentatif et extraction

Le Bundle contient trois cas complémentaires : un patient complet, un patient dont plusieurs champs facultatifs sont absents, et une ressource sans identifiant logique. On reproduit ainsi les cas couverts par les tests sans appeler le serveur public.


In [2]:
bundle = {
    "resourceType": "Bundle",
    "type": "searchset",
    "entry": [
        {"resource": {"resourceType": "Patient", "id": "PAT001",
                      "name": [{"family": "Martin", "given": ["Julie"]}],
                      "gender": "female", "birthDate": "1992-04-03"}},
        {"resource": {"resourceType": "Patient", "id": "PAT002"}},
        {"resource": {"resourceType": "Patient",
                      "name": [{"family": "SansIdentifiant"}]}}
    ]
}

raw_patients = [entry["resource"] for entry in bundle["entry"]]
parsed_patients = [parse_patient(patient) for patient in raw_patients]
valid_patients = [patient for patient in parsed_patients if validate_patient(patient)]

print(f"Ressources reçues : {len(raw_patients)}")
print(f"Patients valides : {len(valid_patients)}")
print(f"Patients rejetés : {len(parsed_patients) - len(valid_patients)}")
valid_patients


Ressources reçues : 3
Patients valides : 2
Patients rejetés : 1


[{'id': 'PAT001',
  'given_name': 'Julie',
  'family_name': 'Martin',
  'gender': 'female',
  'birth_date': '1992-04-03'},
 {'id': 'PAT002',
  'given_name': None,
  'family_name': None,
  'gender': None,
  'birth_date': None}]

> **Observation.** Deux ressources sur trois sont conservées. Les champs facultatifs absents deviennent `None`, tandis que l’absence de `Patient.id` provoque un rejet explicite. Cette distinction évite de confondre donnée incomplète et ressource impossible à identifier.

> **Décision.** Conserver cette validation minimale dans le prototype, mais utiliser un validateur de profils FHIR avant tout contexte de production.


## 3. Persistance et idempotence

On écrit les patients dans SQLite, puis on modifie le prénom de `PAT001` et on réécrit la même clé. Le résultat vérifie le comportement `INSERT OR REPLACE` attendu lors d’une synchronisation répétée.


In [3]:
with TemporaryDirectory() as tmp_dir:
    database_path = Path(tmp_dir) / "patients_demo.db"
    create_database(database_path)
    save_patients(valid_patients, database_path)

    updated_patient = {**valid_patients[0], "given_name": "Julia"}
    save_patients([updated_patient], database_path)
    stored_patients = get_all_patients(database_path)

stored_patients


[('PAT002', None, None, None, None),
 ('PAT001', 'Julia', 'Martin', 'female', '1992-04-03')]

> **Résultat.** SQLite contient toujours deux lignes : `PAT001` a été mis à jour et non dupliqué. La base temporaire est supprimée automatiquement à la fin de la cellule.

## Conclusion et décisions pour la suite

Le pipeline accepte les champs FHIR facultatifs, écarte les ressources sans identifiant et assure une persistance idempotente. Le Notebook 02 vérifie maintenant la transformation amont d’un message HL7 v2 en ressource FHIR.

**Décisions :** conserver les règles déterministes dans Python ; ne jamais dépendre du LLM pour valider ou persister un patient ; ajouter ultérieurement la validation de profils et une stratégie de traçabilité des ressources rejetées.
